# GRPO Tool-Use Agent: Colab quickstart
Teach **Qwen2.5-0.5B-Instruct** to use a calculator tool on GSM8K math problems with reinforcement learning (GRPO), on a free T4 GPU.

**Runtime → Change runtime type → T4 GPU**, then run the cells in order. Training writes checkpoints to Google Drive, so if Colab disconnects, re-run the setup cells and the training cell: it resumes where it stopped.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/grpo-tool-use-agent/runs/qwen05b-grpo'   # checkpoints survive disconnects
RESULTS = '/content/drive/MyDrive/grpo-tool-use-agent/results'

In [ ]:
!git clone -q https://github.com/jayavyas0701/grpo-tool-use-agent.git 2>/dev/null || (cd grpo-tool-use-agent && git pull -q)
%cd /content/grpo-tool-use-agent
!pip -q install -r requirements.txt
!python -m pytest -q tests

## 2. Baseline: the model before any RL
Greedy decoding on 200 held-out GSM8K test problems, with the calculator available. About 5-10 minutes.

In [ ]:
!python evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --limit 200 --name base --out $RESULTS

## 3. GRPO training
4 questions per step x 8 sampled answers each = 32 episodes per step, 150 steps. Expect roughly 1-2 hours on a T4; watch `acc` and `tool use` climb. If Colab disconnects, re-run section 1 and this cell.

In [ ]:
!python train.py --model Qwen/Qwen2.5-0.5B-Instruct --dataset gsm8k --steps 150 --out $RUN_DIR

## 4. Evaluate the trained agent

In [ ]:
!python evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter $RUN_DIR/final_adapter --limit 200 --name grpo --out $RESULTS

## 5. Curves and the before/after table

In [ ]:
!python plot.py --log $RUN_DIR/train_log.csv --summaries $RESULTS/base_summary.json $RESULTS/grpo_summary.json --out $RESULTS
from IPython.display import Image, Markdown, display
display(Image(f'{RESULTS}/training_curves.png'))
display(Markdown(open(f'{RESULTS}/summary.md').read()))

## 6. Look at what the agent writes

In [ ]:
import json
for line in list(open(f'{RESULTS}/grpo_examples.jsonl'))[:3]:
    ex = json.loads(line)
    print('GOLD:', ex['gold'], '| correct:', bool(ex['correct']), '| tool calls:', int(ex['tool_calls']))
    print(ex['completion'][:800]); print('-' * 80)